In [ ]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

In [ ]:


data_dir = r"C:\Users\Aman\shakespeare\data"

rows = []

for file in os.listdir(data_dir):
    if file.endswith("_original.snt.aligned"):

        play = file.replace("_original.snt.aligned", "")
        original_path = os.path.join(data_dir, file)
        modern_path = os.path.join(
            data_dir,
            play + "_modern.snt.aligned"
        )

        if not os.path.exists(modern_path):
            continue

        with open(original_path, "r", encoding="utf-8") as f:
            original = f.readlines()

        with open(modern_path, "r", encoding="utf-8") as f:
            modern = f.readlines()

        for modern_text, shakespeare_text in zip(modern, original):

            modern_text = modern_text.strip()
            shakespeare_text = shakespeare_text.strip()

            if modern_text and shakespeare_text:

                rows.append({
                    "play": play,
                    "modern": modern_text,
                    "shakespeare": shakespeare_text
                })


df = pd.DataFrame(rows)

print(df.shape)
print(df.head())
print(df["play"].value_counts())

In [ ]:
df.head()

In [ ]:
l=df['play'].unique()

length=[]
for i in l:
    length.append(len(df[df['play']==i]))
average_length=sum(length)/len(length)
average_length

    

In [ ]:

plays = df["play"].unique().tolist()

train_plays, temp_plays = train_test_split(
    plays,
    test_size=4,
    random_state=42
)

val_plays, test_plays = train_test_split(
    temp_plays,
    test_size=2,
    random_state=42
)

print("Train plays:", train_plays)
print("Validation plays:", val_plays)
print("Test plays:", test_plays)
train_df = df[df["play"].isin(train_plays)].copy()

val_df = df[df["play"].isin(val_plays)].copy()

test_df = df[df["play"].isin(test_plays)].copy()

In [ ]:
text = "Wherefore art thou"

tokens = tokenizer.encode(text)

print(tokens)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id

In [ ]:
sample = train_df.iloc[0]

print(sample["modern"])
print(sample["shakespeare"])
modern_ids = tokenizer.encode(
    sample["modern"],
    truncation=True,
    max_length=256
)

shakespeare_ids = tokenizer.encode(
    sample["shakespeare"],
    truncation=True,
    max_length=256
)

print(modern_ids)
print(shakespeare_ids)

In [ ]:
MAX_LENGTH = 256

train_inputs = tokenizer(
    train_df["modern"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

train_targets = tokenizer(
    train_df["shakespeare"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

val_inputs = tokenizer(
    val_df["modern"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

val_targets = tokenizer(
    val_df["shakespeare"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

test_inputs = tokenizer(
    test_df["modern"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

test_targets = tokenizer(
    test_df["shakespeare"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

In [ ]:
import torch
from torch.utils.data import Dataset

class ShakespeareDataset(Dataset):

    def __init__(self, inputs, targets):
        self.input_ids = inputs["input_ids"]
        self.target_ids = targets["input_ids"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):

        input_ids = torch.tensor(
            self.input_ids[idx],
            dtype=torch.long
        )

        target_ids = torch.tensor(
            self.target_ids[idx],
            dtype=torch.long
        )

        decoder_input = target_ids.clone()
        decoder_input[1:] = target_ids[:-1]
        decoder_input[0] = tokenizer.eos_token_id

        labels = target_ids

        return {
            "input_ids": input_ids,
            "decoder_input": decoder_input,
            "labels": labels
        }

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_dataset = ShakespeareDataset(train_inputs, train_targets)
val_dataset = ShakespeareDataset(val_inputs, val_targets)
test_dataset = ShakespeareDataset(test_inputs, test_targets)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
#i need to tune this
d_model = 384
n_heads = 6
num_layers = 4
d_ff = 1536
dropout = 0.1
max_length = 256
vocab_size = 50257

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super(EncoderBlock, self).__init__()
        self.n_heads = n_heads
        self.head_size=d_model // n_heads
        self.query = nn.Linear(d_model,d_model,bias=False)
        self.key = nn.Linear(d_model,d_model,bias=False)
        self.value = nn.Linear(d_model,d_model,bias=False)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    def attention(self,x):
        B,T,C=x.shape
        k=self.key(x)
        q=self.query(x)
        v=self.value(x)
        q = q.view(B, T, self.n_heads, self.head_size)
        k = k.view(B, T, self.n_heads, self.head_size)
        v = v.view(B, T, self.n_heads, self.head_size)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        wei=q@k.transpose(-2,-1) / (self.head_size ** 0.5)
        #tril=torch.tril(torch.ones(T,T),device=x.device)
        #wei=wei.masked_fill(tril==0,float('-inf'))
        wei=F.softmax(wei,dim=-1)
        out=wei@v
        out = out.transpose(1, 2)
        out = out.contiguous().view(B, T, C)
        out = self.proj(out)
     
        return out
    def forward(self, x):
        x = x + self.dropout(self.attention(self.ln1(x)))
        x = x + self.ffn(self.ln2(x))
        return x

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super(DecoderBlock, self).__init__()
        self.n_heads = n_heads
        self.head_size=d_model // n_heads
        self.query = nn.Linear(d_model,d_model,bias=False)
        self.key = nn.Linear(d_model,d_model,bias=False)
        self.value = nn.Linear(d_model,d_model,bias=False)
        self.cross_query = nn.Linear(d_model, d_model, bias=False)
        self.cross_key = nn.Linear(d_model, d_model, bias=False)
        self.cross_value = nn.Linear(d_model, d_model, bias=False)
        self.cross_proj = nn.Linear(d_model, d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    def masked_self_attention(self,x):
        B,T,C=x.shape
        k=self.key(x)
        q=self.query(x)
        v=self.value(x)
        q = q.view(B, T, self.n_heads, self.head_size)
        k = k.view(B, T, self.n_heads, self.head_size)
        v = v.view(B, T, self.n_heads, self.head_size)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        wei=q@k.transpose(-2,-1) / (self.head_size ** 0.5)
        tril=torch.tril(torch.ones(T,T),device=x.device)
        wei=wei.masked_fill(tril==0,float('-inf'))
        wei=F.softmax(wei,dim=-1)
        out=wei@v
        out = out.transpose(1, 2)
        out = out.contiguous().view(B, T, C)
        out = self.proj(out)
     
        return out
    def cross_attention(self, x, encoder_output):
        B, T, C = x.shape
        S = encoder_output.shape[1]
        k=self.cross_key(encoder_output)
        q=self.cross_query(x)
        v=self.cross_value(encoder_output)
        q = q.view(B, T, self.n_heads, self.head_size)
        k = k.view(B, S, self.n_heads, self.head_size)
        v = v.view(B, S, self.n_heads, self.head_size)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        wei=q@k.transpose(-2,-1) / (self.head_size ** 0.5)
        wei=F.softmax(wei,dim=-1)
        out=wei@v
        out = out.transpose(1, 2)
        out = out.contiguous().view(B, T, C)
        out = self.cross_proj(out)
        return out
    def forward(self, x, encoder_output):

        x = x + self.dropout(
            self.masked_self_attention(self.ln1(x))
        )

        x = x + self.dropout(
            self.cross_attention(
                self.ln2(x),
                encoder_output
            )
        )

        x = x + self.ffn(self.ln3(x))

        return x

In [ ]:

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_layers, d_ff, dropout, max_length):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)
        
    def forward(self, x,target):    
        B,T=x.shape
        token_emb =self.token_embedding(x)
        positions=torch.arange(T,device=x.device)
        target_emb = self.token_embedding(target)
        target_positions = torch.arange(target.shape[1],device=target.device)
        target_pos_emb = self.position_embedding(target_positions)
        target = target_emb + target_pos_emb
        
        pos_emb = self.position_embedding(positions)

        x = token_emb + pos_emb
        for block in self.blocks:
            x = block(x)
        encoder_output = x
        for block in self.decoder_blocks:
            target = block(target, encoder_output)
        
        

        logits = self.lm_head(target)
        return logits
    

In [ ]:
model = TransformerModel(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    dropout=dropout,
    max_length=max_length
)

In [ ]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"]
decoder_input = batch["decoder_input"]
labels = batch["labels"]

In [ ]:
logits = model(input_ids, decoder_input)

print(logits.shape)
print(labels.shape)